# Todos os serviços na nuvem do escritório (`labdados` + API key)

<a href="https://colab.research.google.com/github/lab-dados/labdados-sdk/blob/main/examples/notebooks/nuvem_todos_os_modelos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Este notebook roda **todos os exemplos que usam a API key do escritório**, variando os modelos disponíveis na nuvem:

| Seção | Função | Modelos |
|---|---|---|
| 3 | `labdados.ocr` | `azure-document-intelligence` (padrão), `mistral-ocr` |
| 4 | `labdados.transcricao` | `azure-speech` (padrão, com diarização) |
| 5 | `labdados.estruturacao` | `gpt-4.1-mini` (padrão), `gpt-4.1`, `gpt-5-mini`, `gpt-5.6-luna`, `gpt-5.6-terra`, `gpt-6-luna`, `DeepSeek-V4-Flash`, `Mistral-Large-3`, `Kimi-K2.6` |
| 6 | `labdados.anonimizacao` | `privacy-filter` (padrão), `lenerbr` |
| 7 | `labdados.embeddings` | `text-embedding-3-small` (padrão), `text-embedding-3-large`, `embed-v-4-0`, `Cohere-embed-v3-multilingual` |
| 8 | `labdados.Client` | conferir a chave e listar suas solicitações |

Os modelos rodam no Azure AI Foundry da FGV, chamados pelo escritório. **Por enquanto a API key é de uso interno da equipe LabDados** — peça a sua no portal do escritório, em
<https://labdados-frontend.livelydesert-3e3e3dd8.brazilsouth.azurecontainerapps.io/consultoria/api-key>.

Cada célula de código é **auto-suficiente** (importa o pacote e pede a chave se ainda não estiver no ambiente): dá para rodar só a seção que interessa.

> **Custo**: cada chamada consome o orçamento do LabDados. Os exemplos usam arquivos pequenos (centavos no total). Os modelos `gpt-5.6-terra` e `gpt-4.1` são os mais caros da lista — em volume, teste antes numa amostra.

## 1. Instalar

In [ ]:
# Versão mais recente direto do GitHub (a do PyPI pode estar atrás).
# Para o modo nuvem não precisa de extras: o processamento roda no escritório.
%pip install -q 'labdados @ git+https://github.com/lab-dados/labdados-sdk'

## 2. Chave e arquivos de exemplo

Guarde a chave em `LABDADOS_API_KEY` (no Colab, prefira os *Secrets* 🔑 da barra lateral e copie com `from google.colab import userdata; os.environ['LABDADOS_API_KEY'] = userdata.get('LABDADOS_API_KEY')`). A célula abaixo confere a chave e baixa/gera os arquivos usados no resto do notebook.

In [ ]:
import os
from getpass import getpass

import labdados

if not os.environ.get('LABDADOS_API_KEY'):
    os.environ['LABDADOS_API_KEY'] = getpass('LABDADOS_API_KEY: ')

import urllib.request


def baixar(url, destino):
    """Baixa com User-Agent identificado e falha alto se vier vazio."""
    req = urllib.request.Request(url, headers={'User-Agent': 'labdados-sdk-notebook/1.0 (https://github.com/lab-dados/labdados-sdk)'})
    with urllib.request.urlopen(req, timeout=120) as resp, open(destino, 'wb') as f:
        f.write(resp.read())
    tamanho = os.path.getsize(destino)
    if tamanho < 1000:
        raise RuntimeError(f'Download de {url} veio vazio ({tamanho} bytes) — tente de novo ou use outro arquivo.')
    print(f'{destino}: {tamanho:,} bytes')


client = labdados.Client(progress=False)
print('Conexão:', client.test_connection())

os.makedirs('dados/pdfs', exist_ok=True)
os.makedirs('dados/textos', exist_ok=True)

# PDF público (paper TraceMonkey, hospedado pelo projeto pdf.js da Mozilla)
baixar('https://mozilla.github.io/pdf.js/web/compressed.tracemonkey-pldi-09.pdf', 'dados/pdfs/tracemonkey.pdf')
# Áudio curto em domínio público (discurso de posse de J. F. Kennedy, 1961),
# hospedado no repositório do Whisper no GitHub
baixar('https://raw.githubusercontent.com/openai/whisper/main/tests/jfk.flac', 'dados/audio.flac')


# Ementas fictícias para estruturação e embeddings
import pandas as pd

pd.DataFrame({
    'id': [1, 2, 3, 4],
    'ementa': [
        'APELAÇÃO CÍVEL. Plano de saúde. TJSP, 2022. Negativa de cobertura para tratamento oncológico. '
        'Pedido julgado procedente. Cobertura obrigatória. Recurso desprovido.',
        'AGRAVO DE INSTRUMENTO. Direito do consumidor. TJRJ, 2021. Cobrança indevida em fatura de telefonia. '
        'Pedido julgado improcedente por ausência de prova.',
        'APELAÇÃO. Família. TJMG, 2023. Pedido de guarda compartilhada julgado procedente em parte. '
        'Melhor interesse da criança.',
        'APELAÇÃO CÍVEL. Plano de saúde. TJSP, 2023. Recusa de custeio de cirurgia bariátrica. '
        'Pedido julgado procedente. Abusividade da negativa.',
    ],
}).to_csv('dados/ementas.csv', index=False)

with open('dados/textos/peticao.txt', 'w', encoding='utf-8') as f:
    f.write(
        'O autor João Pereira da Silva, CPF 123.456.789-00, residente na Rua das Flores, 100, '
        'São Paulo/SP, e-mail joao.silva@example.com, telefone (11) 98765-4321, ajuizou ação em '
        '10/03/2024 contra a Empresa Alfa Ltda., representada por Maria Oliveira.'
    )
!ls -R dados

## 3. OCR — `labdados.ocr`

- **`azure-document-intelligence`** (padrão): OCR da Microsoft, processamento no Brasil. Texto puro.
- **`mistral-ocr`**: devolve markdown, preservando títulos e tabelas. Processamento global.

`text=True` devolve o texto direto (os arquivos continuam em `saida`). A primeira chamada depois de um tempo parado pode levar alguns minutos (o servidor "acorda").

In [ ]:
import os
from getpass import getpass

import labdados

if not os.environ.get('LABDADOS_API_KEY'):
    os.environ['LABDADOS_API_KEY'] = getpass('LABDADOS_API_KEY: ')

for modelo in ['azure-document-intelligence', 'mistral-ocr']:
    texto = labdados.ocr(
        'dados/pdfs/',
        modelo=modelo,
        formato='md',
        saida=f'resultados/ocr_{modelo}/',
        text=True,
    )
    print(f'===== {modelo}: {len(texto):,} caracteres =====')
    print(texto[:600], '\n')

## 4. Transcrição — `labdados.transcricao`

**`azure-speech`** (padrão): transcrição da Microsoft com diarização (separa falantes como `SPEAKER_00`, `SPEAKER_01`…), processamento no Brasil. Formatos `srt` (padrão), `vtt` ou `txt`. Use `idioma='pt'` para português; aqui o áudio de exemplo (Kennedy, 1961) é em inglês, então usamos `'auto'`.

In [ ]:
import os
from getpass import getpass

import labdados

if not os.environ.get('LABDADOS_API_KEY'):
    os.environ['LABDADOS_API_KEY'] = getpass('LABDADOS_API_KEY: ')

srt = labdados.transcricao(
    'dados/audio.flac',
    modelo='azure-speech',
    idioma='auto',
    diarizacao=True,
    formato='srt',
    saida='resultados/transcricao/',
    text=True,
)
print(srt)

## 5. Estruturação — `labdados.estruturacao`

Extrai campos de cada documento segundo um JSON Schema. Aqui rodamos **os 9 modelos** sobre as mesmas ementas e comparamos numa tabela. Os GPT-5/6 são modelos de raciocínio (ignoram `temperatura`).

In [ ]:
import os
from getpass import getpass

import labdados

if not os.environ.get('LABDADOS_API_KEY'):
    os.environ['LABDADOS_API_KEY'] = getpass('LABDADOS_API_KEY: ')

import glob
import zipfile

import pandas as pd

schema = {
    'type': 'object',
    'properties': {
        'tribunal': {'type': 'string', 'description': 'Sigla do tribunal (TJSP, TJRJ, STJ...)'},
        'ano': {'type': 'integer'},
        'tema': {'type': 'string', 'description': 'Área: saúde, consumidor, família...'},
        'resultado': {'type': 'string', 'enum': ['procedente', 'improcedente', 'parcialmente procedente']},
    },
}

MODELOS = [
    'gpt-4.1-mini', 'gpt-4.1', 'gpt-5-mini', 'gpt-5.6-luna', 'gpt-5.6-terra',
    'gpt-6-luna', 'DeepSeek-V4-Flash', 'Mistral-Large-3', 'Kimi-K2.6',
]

linhas = []
for modelo in MODELOS:
    saida = f'resultados/estruturacao_{modelo}/'
    labdados.estruturacao(
        'dados/ementas.csv',
        coluna_texto='ementa',
        schema=schema,
        prompt_sistema='Você extrai metadados de ementas de acórdãos brasileiros.',
        modelo=modelo,
        saida=saida,
    )
    with zipfile.ZipFile(sorted(glob.glob(saida + '*.zip'))[-1]) as zf:
        df = pd.read_csv(zf.open('consolidated.csv'))
    df.insert(0, 'modelo', modelo)
    linhas.append(df)

comparacao = pd.concat(linhas, ignore_index=True)
comparacao.pivot_table(index='_source_doc', columns='modelo', values='resultado', aggfunc='first')

## 6. Anonimização — `labdados.anonimizacao`

Detecta e mascara dados pessoais. **Não é garantia de anonimização** — sempre revise o resultado.

- **`privacy-filter`** (padrão): multilíngue, 8 categorias (pessoa, e-mail, telefone, endereço…).
- **`lenerbr`**: BERT treinado em decisões judiciais brasileiras (pessoa, organização, local, tempo, legislação, jurisprudência).

In [ ]:
import os
from getpass import getpass

import labdados

if not os.environ.get('LABDADOS_API_KEY'):
    os.environ['LABDADOS_API_KEY'] = getpass('LABDADOS_API_KEY: ')

import glob
import zipfile

for modelo in ['privacy-filter', 'lenerbr']:
    saida = f'resultados/anonimizacao_{modelo}/'
    labdados.anonimizacao('dados/textos/', modelo=modelo, estrategia='categoria', saida=saida)
    with zipfile.ZipFile(sorted(glob.glob(saida + '*.zip'))[-1]) as zf:
        nome = next(n for n in zf.namelist() if n.endswith('.txt'))
        print(f'===== {modelo} =====')
        print(zf.read(nome).decode('utf-8'), '\n')

## 7. Embeddings — `labdados.embeddings`

Transforma cada texto em um vetor: textos de sentido parecido ficam com vetores próximos (busca semântica, agrupamento, classificação). Com `dataframe=True` volta um `pandas.DataFrame` com a coluna `embedding`.

Abaixo, para cada modelo, a similaridade de cosseno entre as ementas: as duas de **plano de saúde** (1 e 4) devem ser as mais parecidas entre si.

> Use sempre o mesmo modelo para os textos que serão comparados — vetores de modelos diferentes não são comparáveis.

In [ ]:
import os
from getpass import getpass

import labdados

if not os.environ.get('LABDADOS_API_KEY'):
    os.environ['LABDADOS_API_KEY'] = getpass('LABDADOS_API_KEY: ')

import numpy as np
import pandas as pd

for modelo in ['text-embedding-3-small', 'text-embedding-3-large', 'embed-v-4-0', 'Cohere-embed-v3-multilingual']:
    df = labdados.embeddings(
        'dados/ementas.csv',
        coluna_texto='ementa',
        modelo=modelo,
        saida=f'resultados/embeddings_{modelo}/',
        dataframe=True,
    )
    v = np.array(df['embedding'].tolist())
    v = v / np.linalg.norm(v, axis=1, keepdims=True)
    sim = pd.DataFrame(v @ v.T, index=df['doc_id'], columns=df['doc_id']).round(2)
    print(f'===== {modelo} ({v.shape[1]} dimensões) =====')
    print(sim, '\n')

## 8. Cliente — conferir a chave e listar solicitações

O `labdados.Client` reaproveita a mesma chave em várias chamadas (`client.ocr(...)`, `client.estruturacao(...)` etc.) e lista suas solicitações — as mesmas que aparecem em **Minhas solicitações** no portal.

In [ ]:
import os
from getpass import getpass

import labdados

if not os.environ.get('LABDADOS_API_KEY'):
    os.environ['LABDADOS_API_KEY'] = getpass('LABDADOS_API_KEY: ')

import pandas as pd

client = labdados.Client(progress=False)
print('Conexão:', client.test_connection())

pd.DataFrame(client.solicitacoes(limite=20))[['created_at', 'service_id', 'model_id', 'status']]

## Próximos passos

- Documentação: <https://lab-dados.github.io/labdados-sdk/>
- Modo local (sem API key, roda no seu computador): <https://lab-dados.github.io/labdados-sdk/modo-local.html>
- Notebooks por serviço: [`examples/notebooks/`](https://github.com/lab-dados/labdados-sdk/tree/main/examples/notebooks)